# idk feels right for this assignment

## constants n stuff

In [1]:
import numpy as np
from tqdm import tqdm

In [2]:
MOTIF_LEN = 20
CHAR_TO_INT = {"A": 0, "C": 1, "G": 2, "T": 3}

## io

In [3]:
def file_to_genomes(name: str) -> np.ndarray[int]:
    ret = []
    curr = []
    with open(name) as read:
        for line in read:
            if line[0] == ">":
                if curr:
                    ret.append([CHAR_TO_INT[c] for c in "".join(curr)])
                curr = []
                continue
            curr.append(line.strip())
    if curr:
        ret.append([CHAR_TO_INT[c] for c in "".join(curr)])

    return np.array(ret, dtype=int)

In [4]:
genomes = file_to_genomes("data/boundcentered.fasta")
predict = file_to_genomes("data/boundrandomoffset.fasta")
len_ = genomes.shape[1]

## motif gen

didn't really wanna wait for the prof to cover it, so just used some slides
i found [here](https://www.biostat.wisc.edu/~craven/776/lecture9.pdf) instead

In [5]:
# these two functions infer relevant parameters from the input shapes
def estimate_pos(seqs: np.ndarray[int], pwm: np.ndarray[float]) -> np.ndarray[float]:
    kmer_len = pwm.shape[0]
    kmer_num = seqs.shape[1] - kmer_len + 1
    pos_prob = np.ones((seqs.shape[0], kmer_num))
    
    m_range = np.arange(kmer_len)
    for i, s in enumerate(seqs):
        for j, p in enumerate(range(kmer_num)):
            pos_prob[i][j] = pwm[m_range, s[p:p+kmer_len]].prod()
    
    pos_prob /= pos_prob.sum(axis=1)[:, None]
    return pos_prob


def best_pwm(seqs: np.ndarray[int], pos_prob: np.ndarray[float]) -> np.ndarray[float]:
    kmer_num = pos_prob.shape[1]
    motif_len = seqs.shape[1] - kmer_num + 1
    pwm = np.zeros((motif_len, 4))
    for val in range(4):
        sel_ind = seqs == val
        for pos in range(motif_len):
            pwm[pos][val] = pos_prob[sel_ind[:, pos : pos + kmer_num]].sum() + 1
    pwm /= pwm.sum(axis=1)[:, None]
    return pwm

In [ ]:
guesses = np.random.uniform(0, len_ - MOTIF_LEN + 1, size=len(genomes)).astype(int)
pos = np.zeros((genomes.shape[0], genomes.shape[1] - MOTIF_LEN + 1))
pos[np.arange(len(pos)), guesses] = 1

iter_amt = 100
for _ in tqdm(range(iter_amt)):
    pwm = best_pwm(genomes, pos)
    pos = estimate_pos(genomes, pwm)

100%|██████████| 100/100 [16:10<00:00,  9.71s/it]


## predictions

In [7]:
inds = estimate_pos(predict, pwm).argmax(axis=1)
output = [f"seq{v + 1},{ind + MOTIF_LEN // 2}" for v, ind in enumerate(inds)]

In [8]:
print("\n".join(output), file=open("predictions.csv", "w"))